In [ ]:
!mkdir -p megha data checkpoints
!pip install transformers tokenizers accelerate huggingface_hub


In [ ]:
%%writefile megha/__init__.py



In [ ]:
%%writefile megha/config.py
from dataclasses import dataclass

@dataclass
class MeghaConfig:
    vocab_size: int = 8000
    max_seq_len: int = 512
    d_model: int = 512
    n_layers: int = 8
    n_heads: int = 8
    dropout: float = 0.1
    batch_size: int = 16
    learning_rate: float = 3e-4
    epochs: int = 3



In [ ]:
%%writefile megha/model.py
import torch
import torch.nn as nn
from .config import MeghaConfig

class MeghaBlock(nn.Module):
    def __init__(self, config: MeghaConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=config.d_model, 
            num_heads=config.n_heads, 
            dropout=config.dropout,
            batch_first=True
        )
        self.ln_2 = nn.LayerNorm(config.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model),
            nn.Dropout(config.dropout)
        )

    def forward(self, x, attention_mask=None):
        B, T, C = x.shape
        attn_mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        
        attn_out, _ = self.attn(
            self.ln_1(x), self.ln_1(x), self.ln_1(x), 
            attn_mask=attn_mask, need_weights=False, is_causal=True
        )
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return x

class MeghaModel(nn.Module):
    def __init__(self, config: MeghaConfig):
        super().__init__()
        self.config = config
        
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        
        self.blocks = nn.Sequential(*[MeghaBlock(config) for _ in range(config.n_layers)])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        
        # Weight tying
        self.token_emb.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        
        x = self.token_emb(idx) + self.pos_emb(pos)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.config.vocab_size), targets.view(-1))
            
        return logits, loss
        
    def generate(self, idx, max_new_tokens):
        # Basic greedy generation for testing
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] # focus on last time step
            probs = torch.nn.functional.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



In [ ]:
%%writefile megha/tokenizer.py
import os
import json
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from .config import MeghaConfig

class MeghaTokenizer:
    def __init__(self, config: MeghaConfig):
        self.config = config
        self.tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
        self.tokenizer.pre_tokenizer = Whitespace()
        self.trainer = BpeTrainer(
            vocab_size=config.vocab_size, 
            special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
        )
        
    def train_from_iterator(self, iterator):
        self.tokenizer.train_from_iterator(iterator, self.trainer)
        
    def save(self, path):
        self.tokenizer.save(path)
        
    def load(self, path):
        self.tokenizer = Tokenizer.from_file(path)
        
    def encode(self, text):
        return self.tokenizer.encode(text).ids
        
    def decode(self, ids):
        return self.tokenizer.decode(ids)

if __name__ == "__main__":
    # Script to train the tokenizer on generated curriculum data
    print("Training Tokenizer...")
    config = MeghaConfig()
    megha_tok = MeghaTokenizer(config)
    
    # 1. Load data from the data folder
    data_path = "data/level_0_curriculum.json"
    if not os.path.exists(data_path):
        print(f"Error: Data file {data_path} not found. Run data_gen.py first.")
        exit(1)
        
    # 2. Extract text into an iterator from all available curriculum files
    def text_iterator():
        import glob
        files = glob.glob("data/level_*_curriculum.json")
        for file_path in files:
            with open(file_path, "r", encoding="utf-8") as f:
                dataset = json.load(f)
                for item in dataset:
                    yield item["text"]
            
    # 3. Train
    megha_tok.train_from_iterator(text_iterator())
    
    # 4. Save
    os.makedirs("data", exist_ok=True)
    megha_tok.save("data/tokenizer.json")
    print(f"Tokenizer trained and saved to data/tokenizer.json with vocab size: {megha_tok.tokenizer.get_vocab_size()}")
    
    # Quick Test
    test_text = "The computer is on."
    encoded = megha_tok.encode(test_text)
    print(f"\nTest string: '{test_text}'")
    print(f"Encoded IDs: {encoded}")
    print(f"Decoded: '{megha_tok.decode(encoded)}'")



In [ ]:
%%writefile megha/dataset.py
import json
import torch
from torch.utils.data import Dataset, DataLoader
from .tokenizer import MeghaTokenizer
from .config import MeghaConfig
import os

class MeghaDataset(Dataset):
    def __init__(self, data_path: str, tokenizer: MeghaTokenizer, config: MeghaConfig):
        self.config = config
        self.tokenizer = tokenizer
        
        if not os.path.exists(data_path):
            raise FileNotFoundError(f"{data_path} not found. Run data_gen.py first.")
            
        with open(data_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
            
        # Saare text sentences ko encode karke ek long sequence banayenge
        all_tokens = []
        for item in raw_data:
            text = item["text"]
            tokens = self.tokenizer.encode(text)
            all_tokens.extend(tokens)
            
        # Agar data chhota hai (jaise abhi 5 sentences hain), 
        # toh max_seq_len ko temporary chhota kar dete hain warning se bachne ke liye
        if len(all_tokens) <= self.config.max_seq_len:
            print(f"Warning: Data size ({len(all_tokens)}) is smaller than max_seq_len ({self.config.max_seq_len}). Padding with 0s.")
            pad_len = self.config.max_seq_len - len(all_tokens) + 2
            all_tokens.extend([0] * pad_len)
            
        self.data = torch.tensor(all_tokens, dtype=torch.long)
        
    def __len__(self):
        return len(self.data) - self.config.max_seq_len - 1
        
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.config.max_seq_len]
        y = self.data[idx + 1 : idx + self.config.max_seq_len + 1]
        return x, y

def get_dataloader(data_path: str, tokenizer_path: str, config: MeghaConfig):
    tokenizer = MeghaTokenizer(config)
    
    if os.path.exists(tokenizer_path):
        tokenizer.load(tokenizer_path)
    else:
        raise FileNotFoundError(f"Tokenizer not found at {tokenizer_path}. Run tokenizer.py first.")
        
    dataset = MeghaDataset(data_path, tokenizer, config)
    dataloader = DataLoader(
        dataset, 
        batch_size=config.batch_size, 
        shuffle=True, 
        drop_last=True
    )
    return dataloader, tokenizer



In [ ]:
%%writefile megha/data_gen.py
import json
import argparse
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Prompts for different levels of curriculum
LEVEL_PROMPTS = {
    0: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 0 - Basic English Grammar and Vocabulary.
Generate 50 simple training examples covering basic sentence structure, nouns, verbs, pronouns, and basic reasoning (e.g., 'The server is running', 'it -> EC2').
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Example: [{"text": "The server is running."}, {"text": "Is the database active?"}]
Output nothing but the JSON array. Do not include markdown blocks like ```json.""",
    
    1: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 1 - General Knowledge & Basic Reasoning.
Generate 50 training examples covering: Numbers (counting, comparison), Time (seconds, hours), Common Concepts (input, output, process), and Basic Reasoning (e.g. 'If a server is powered off, it cannot serve requests.').
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks.""",

    2: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 2 - Computer Fundamentals.
Generate 50 training examples covering: Computer Architecture (CPU, ALU, RAM), Memory & Storage (virtual memory, HDD vs SSD), Operating Systems (kernel, system calls, threads), and Basic Programming Concepts.
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field containing factual, clear statements.
Output nothing but the JSON array. Do not include markdown blocks.""",

    3: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 3 - Linux Operating System.
Generate 50 training examples covering: Linux Filesystem (/, /etc, /var), Essential Commands (ls, mkdir, grep, chmod), Processes & Services (ps, kill, systemctl), and Networking (ping, curl).
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field containing factual, clear statements.
Output nothing but the JSON array. Do not include markdown blocks.""",

    4: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 4 - Networking.
Generate training examples covering: TCP/IP, OSI Model, Subnetting (CIDR), DNS, HTTP Status Codes, and common Ports (22, 80, 443).
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field containing factual, clear statements.
Output nothing but the JSON array. Do not include markdown blocks.""",

    5: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 5 - Cloud Computing Fundamentals.
Generate training examples covering: Virtualization (hypervisors, VMs), Cloud Models (IaaS, PaaS, SaaS), Deployment Models (public, private, hybrid), and Cloud Characteristics (elasticity, high availability).
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field containing factual, clear statements.
Output nothing but the JSON array. Do not include markdown blocks.""",

    6: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 6 - AWS Core.
Generate training examples covering: AWS EC2 (instances, AMIs), S3 (buckets, objects), IAM (roles, policies), VPC (subnets, internet gateways), and RDS.
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field containing factual, clear statements.
Output nothing but the JSON array. Do not include markdown blocks.""",

    7: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 7 - Docker & Containers.
Generate training examples covering: Docker fundamentals (images, containers, Dockerfile), commands (docker build, docker run, docker ps), and Docker networking/volumes.
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field containing factual, clear statements.
Output nothing but the JSON array. Do not include markdown blocks.""",

    8: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 8 - Kubernetes.
Generate training examples covering: K8s fundamentals (cluster, pods, deployments, services), networking (ClusterIP, NodePort, Ingress), and scaling (HPA).
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field containing factual, clear statements.
Output nothing but the JSON array. Do not include markdown blocks.""",

    9: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 9 - DevOps & CI/CD.
Generate training examples covering: Git operations, CI/CD pipelines, and Infrastructure as Code (Terraform).
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field containing factual, clear statements.
Output nothing but the JSON array. Do not include markdown blocks.""",

    10: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 10 - Cloud Security.
Generate training examples covering: Authentication, IAM policies, Zero Trust, KMS encryption, and common vulnerabilities (public S3, open ports).
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field containing factual, clear statements.
Output nothing but the JSON array. Do not include markdown blocks.""",

    11: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 11 - Cloud Troubleshooting.
Generate training scenarios that follow a symptom->investigation->cause->solution pattern (e.g., 'EC2 is unreachable', 'Website returns 502', 'S3 AccessDenied').
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field describing the troubleshooting steps.
Output nothing but the JSON array. Do not include markdown blocks.""",

    12: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 12 - Cloud Architecture.
Generate training examples describing system designs: Highly Available architectures, Scalable web apps (Load Balancer + Auto Scaling), Serverless (API Gateway + Lambda), and Event-driven systems.
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field describing the architecture.
Output nothing but the JSON array. Do not include markdown blocks.""",

    13: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 13 - Cloud Reasoning.
Generate scenario-based training examples. Present a scenario (e.g., '10,000 users suddenly access the site') and explain the expected system behavior and the solution.
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field containing the scenario and reasoning.
Output nothing but the JSON array. Do not include markdown blocks.""",

    14: """You are an expert AI teacher generating curriculum data for a smaller language model.
Topic: Level 14 - Multi-step Problem Solving.
Generate advanced training examples showing step-by-step problem resolution: Identify symptoms -> Collect evidence -> Generate hypotheses -> Test -> Fix -> Verify.
Format the output STRICTLY as a JSON array of objects, with each object having a "text" field.
Output nothing but the JSON array. Do not include markdown blocks."""
}

def generate_curriculum_real(level: int):
    print(f"Loading Qwen model for Level {level} curriculum generation...")
    # Upgraded Teacher to 3 Billion Parameters for much smarter data generation
    model_id = "Qwen/Qwen2.5-3B-Instruct"  
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    prompt = LEVEL_PROMPTS.get(level, LEVEL_PROMPTS[0])
    
    messages = [
        {"role": "system", "content": "You are a highly structured data generation AI."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    all_data = []
    target_examples = 1000  # We want 1000 high-quality examples per level
    batch_size = 50         # Qwen will generate 50 at a time
    
    print(f"Teacher is generating {target_examples} examples for Level {level} (in batches)...")
    
    iterations = target_examples // batch_size
    for i in range(iterations):
        print(f"Generating batch {i+1}/{iterations}...")
        try:
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=2048,
                temperature=0.8,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
            
            generated_ids = [
                output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]
            
            response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
            # Parse JSON
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0]
            elif "```" in response:
                response = response.split("```")[1].split("```")[0]
                
            data = json.loads(response.strip())
            
            # Ensure it's a list
            if isinstance(data, list):
                all_data.extend(data)
            elif isinstance(data, dict) and "text" in data:
                all_data.append(data)
                
            print(f"Current total for Level {level}: {len(all_data)} examples.")
            
        except Exception as e:
            print(f"Batch {i+1} failed to parse or generate, skipping. Error: {e}")
            
    print(f"Successfully generated {len(all_data)} high-quality examples for Level {level}!")
    return all_data

def generate_curriculum_dummy(level: int):
    print(f"Generating DUMMY curriculum for Level {level} (Local PC Test)...")
    simulated_response = [
        {"text": "The computer is on."},
        {"text": "A network connects devices."},
        {"text": "She types on the keyboard."},
        {"text": "Data is stored in memory."},
        {"text": "He clicks the mouse."}
    ] * 100
    return simulated_response

def save_curriculum(data, level):
    os.makedirs("data", exist_ok=True)
    file_path = f"data/level_{level}_curriculum.json"
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)
    print(f"Curriculum saved to {file_path} successfully!")

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Generate MEGHA Curriculum using Qwen")
    parser.add_argument("--level", type=int, default=0, help="Curriculum level to generate")
    parser.add_argument("--real", action="store_true", help="Use actual HuggingFace Qwen model (requires GPU)")
    args = parser.parse_args()
    
    if args.real:
        generated_data = generate_curriculum_real(args.level)
    else:
        generated_data = generate_curriculum_dummy(args.level)
        
    save_curriculum(generated_data, args.level)



In [ ]:
%%writefile megha/train.py
import torch
import torch.optim as optim
from .model import MeghaModel
from .config import MeghaConfig
from .dataset import get_dataloader
import time
import os

def train_level(level: int):
    print(f"Starting Training for Level {level}...")
    config = MeghaConfig()
    
    # Kaggle par jab GPU hoga toh yahan automatically 'cuda' select ho jayega
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model = MeghaModel(config).to(device)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model Parameters: {num_params / 1e6:.2f} M")
    
    data_path = f"data/level_{level}_curriculum.json"
    tokenizer_path = "data/tokenizer.json"
    
    try:
        dataloader, tokenizer = get_dataloader(data_path, tokenizer_path, config)
    except FileNotFoundError as e:
        print(e)
        return
        
    print(f"Dataset loaded. Total batches per epoch: {len(dataloader)}")
    if len(dataloader) == 0:
        print("Data is too small for the batch size! Try generating more sentences in data_gen.py")
        return
        
    optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)
    
    model.train()
    for epoch in range(config.epochs):
        print(f"\n--- Epoch {epoch+1}/{config.epochs} ---")
        
        for step, (x, y) in enumerate(dataloader):
            t0 = time.time()
            
            x, y = x.to(device), y.to(device)
            
            optimizer.zero_grad()
            logits, loss = model(x, targets=y)
            loss.backward()
            optimizer.step()
            
            dt = time.time() - t0
            
            if step % 10 == 0 or step == len(dataloader) - 1:
                print(f"Step {step} | Loss: {loss.item():.4f} | Time: {dt*1000:.2f}ms")
                
    # Save checkpoint weights for inference later
    os.makedirs("checkpoints", exist_ok=True)
    checkpoint_path = f"checkpoints/megha_level_{level}.pt"
    torch.save(model.state_dict(), checkpoint_path)
    print(f"\nTraining complete. Model weights saved to {checkpoint_path}")

if __name__ == "__main__":
    train_level(0)



In [ ]:
%%writefile megha/evaluate.py
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from .model import MeghaModel
from .config import MeghaConfig
from .tokenizer import MeghaTokenizer
import os

def run_evaluation():
    print("Starting MEGHA Evaluation Phase (Level 15)...")
    
    # Load MEGHA
    config = MeghaConfig()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    megha_model = MeghaModel(config).to(device)
    
    # Load the latest checkpoint (Level 14)
    ckpt_path = "checkpoints/megha_level_14.pt"
    if os.path.exists(ckpt_path):
        megha_model.load_state_dict(torch.load(ckpt_path, map_location=device))
        print(f"Loaded MEGHA from {ckpt_path}")
    else:
        print("Warning: Level 14 checkpoint not found. Using untrained MEGHA for testing.")
        
    megha_model.eval()
    megha_tok = MeghaTokenizer(config)
    megha_tok.load("data/tokenizer.json")
    
    # Load Qwen (Teacher)
    print("Loading Teacher (Qwen 3B) for grading...")
    teacher_id = "Qwen/Qwen2.5-3B-Instruct"
    teacher_tok = AutoTokenizer.from_pretrained(teacher_id)
    teacher = AutoModelForCausalLM.from_pretrained(
        teacher_id, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )
    
    # 5 Sample Questions covering the syllabus
    test_questions = {
        "Level 3 (Linux)": "What is the command to change file permissions in Linux?",
        "Level 6 (AWS)": "What is Amazon EC2 used for?",
        "Level 7 (Docker)": "What does a Dockerfile do?",
        "Level 10 (Security)": "Why should you not store AWS access keys in a public S3 bucket?",
        "Level 11 (Troubleshooting)": "If a website returns a 502 error, what could be the problem?"
    }
    
    results = {}
    
    for topic, question in test_questions.items():
        print(f"\n[Testing {topic}] Question: {question}")
        
        # 1. MEGHA generates an answer
        prompt = f"Q: {question}\nA:"
        input_ids = megha_tok.encode(prompt)
        
        # If tokenizer returns empty or very short, pad it safely
        if len(input_ids) == 0:
            input_ids = [0]
            
        x = torch.tensor([input_ids], dtype=torch.long).to(device)
        
        # Generate 20 tokens (Megha is still a small model, so answers might be brief/fragmented)
        with torch.no_grad():
            for _ in range(25):
                logits, _ = megha_model(x)
                next_token = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(0)
                x = torch.cat((x, next_token), dim=1)
                    
        megha_answer = megha_tok.decode(x[0].tolist())
        # Clean up the output string
        megha_answer = megha_answer.replace(prompt, "").strip()
        print(f"MEGHA's Answer: {megha_answer}")
        
        # 2. Qwen grades the answer
        grade_prompt = f"""You are grading an AI student. 
Question: {question}
Student's Answer: {megha_answer}
Rate the student's answer out of 100 based on accuracy and relevance. Only output the numeric score (e.g., 85). If the answer is gibberish, output 0."""
        
        messages = [
            {"role": "system", "content": "You are a strict grader. Output only a number between 0 and 100."},
            {"role": "user", "content": grade_prompt}
        ]
        
        text = teacher_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = teacher_tok([text], return_tensors="pt").to(teacher.device)
        
        generated_ids = teacher.generate(
            **model_inputs,
            max_new_tokens=10,
            temperature=0.1
        )
        
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        score_text = teacher_tok.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        # Ensure score is just digits
        score_text = ''.join(filter(str.isdigit, score_text))
        if not score_text: score_text = "0"
            
        print(f"Teacher's Grade: {score_text}/100")
        results[topic] = score_text
        
    print("\n" + "="*40)
    print("MEGHA FINAL REPORT CARD")
    print("="*40)
    for topic, score in results.items():
        print(f"{topic}: {score}%")
    print("="*40)

if __name__ == "__main__":
    run_evaluation()



In [ ]:
!python megha/data_gen.py --level 0 --real
!python megha/data_gen.py --level 1 --real
!python megha/data_gen.py --level 2 --real
!python megha/data_gen.py --level 3 --real
!python megha/data_gen.py --level 4 --real
!python megha/data_gen.py --level 5 --real
!python megha/data_gen.py --level 6 --real
!python megha/data_gen.py --level 7 --real
!python megha/data_gen.py --level 8 --real
!python megha/data_gen.py --level 9 --real
!python megha/data_gen.py --level 10 --real
!python megha/data_gen.py --level 11 --real
!python megha/data_gen.py --level 12 --real
!python megha/data_gen.py --level 13 --real
!python megha/data_gen.py --level 14 --real
!python -m megha.tokenizer
!python -c "from megha.train import train_level; [train_level(i) for i in range(15)]"
!python -m megha.evaluate
!cp -r checkpoints/* /kaggle/working/ || true
